# 23 — Embedding Benchmarks
**Goal:** Compare Word2Vec, GloVe, and Sentence Transformers on resume-specific tasks.

Chapters 19–22 each claimed their embeddings capture "meaning". This chapter stops trusting claims: it defines a tiny hand-labeled benchmark of resume-skill pairs, runs every model through the same pairs, and lets cosine similarity speak. The benchmark is mini by design — six pairs, a handful of models — but the *methodology* is the deliverable: fixed pairs, expected ratings, one scoring function.

**Why it matters for resumes / ATS:** an embedding choice is a product decision. Sentence Transformers cost a ~90 MB model and CPU time per call; GloVe is instant but weak on phrases and out-of-vocabulary words. Measuring both on *resume-specific* pairs (acronym vs full form, related tools, unrelated words) tells you which failure modes you are buying into before you ship the matcher.

## 1. Building a Mini Benchmark

A benchmark is just: fixed inputs, expected outputs, and a scoring rule. The test pairs here encode the failure modes a resume matcher actually faces — the same skill in different casing ("python" vs "Python programming"), related-but-different ("tensorflow" vs "deep learning"), acronym vs full form ("nlp" vs "natural language processing"), and a hard negative ("python" vs "cooking").

**What the code does:** `test_pairs` holds `(w1, w2, expected_rating)` tuples with ratings from 0.0 (unrelated) to 0.9 (same skill). `evaluate_embeddings()` looks both words up in a dict of vectors, computes `cosine_similarity`, and appends only pairs where both words exist — missing words are **skipped silently by design**. The cell prints a status line; note the helper is scaffold — §3's real loop duplicates this logic inline rather than calling `evaluate_embeddings`.

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Test pairs: (resume_skill, job_requirement, expected_similarity_rating)
# 1.0 = same meaning, 0.0 = unrelated
test_pairs = [
    ("python", "Python programming", 0.9),     # same skill
    ("python", "Java", 0.2),                    # different language
    ("tensorflow", "deep learning", 0.7),       # related
    ("nlp", "natural language processing", 0.8), # acronym vs full
    ("docker", "kubernetes", 0.6),              # related tools
    ("python", "cooking", 0.0),                 # unrelated
]

def evaluate_embeddings(embeddings_dict, model_name, test_pairs):
    """Score how well embeddings capture semantic similarity."""
    scores = []
    for w1, w2, expected in test_pairs:
        try:
            v1 = embeddings_dict[w1] if w1 in embeddings_dict else None
            v2 = embeddings_dict[w2] if w2 in embeddings_dict else None
            if v1 is not None and v2 is not None:
                sim = cosine_similarity([v1], [v2])[0][0]
                scores.append((w1, w2, sim, expected))
        except:
            pass
    return scores

print("Benchmark ready. Test pairs defined. Each model will be evaluated on semantic similarity accuracy.")
print("\nNote: We need pre-trained vectors loaded to run this. Check each model's availability.")

## 2. Loading All Models

The benchmark needs every embedding family behind one dict so the scoring loop treats them uniformly. Note what this cell actually loads — and what it doesn't: Sentence Transformers and GloVe. **Word2Vec is named in the chapter title but never loaded here** — gensim's `word2vec-google-news-300` is a ~1.6 GB download, so the notebook skips it and the Summary's "Word2Vec" refers conceptually to Ch. 19's locally trained toy model.

**What the code does:** two `try/except` blocks append to `models`:
- `SentenceTransformer("all-MiniLM-L6-v2")` under the key `"SentenceTransformer"` (needs `pip install sentence-transformers`);
- `api.load("glove-twitter-25")` under its model name (first use downloads ~105 MB).

It finally prints how many models made it in. Expected: **2** when both dependencies are installed, fewer otherwise — the guards exist so a missing library degrades to a warning, not a crash.

In [ ]:
models = {}
# Sentence Transformers
try:
    from sentence_transformers import SentenceTransformer
    st = SentenceTransformer("all-MiniLM-L6-v2")
    models["SentenceTransformer"] = st
    print("✓ SentenceTransformer loaded")
except: print("  SentenceTransformer not available")

# Word2Vec/GloVe via gensim
try:
    import gensim.downloader as api
    # Use a small model if available
    models["glove-twitter-25"] = api.load("glove-twitter-25")
    print("✓ GloVe loaded")
except: print("  GloVe not downloaded (needs internet/first-time download)")

print(f"\nLoaded {len(models)} models for benchmark")

## 3. Running the Benchmark

The loop dispatches on API shape: sentence transformers expose `.encode()` (text in, vector out); gensim keyed vectors are indexed directly with `model[w]` (word in, vector out). Both produce a vector, so the same `cosine_similarity` call scores them.

**What the code does:** for each model, for each pair: encode/index both words, cosine, print with the expected rating. Two behaviors to expect:

- **Sentence Transformer handles everything** — it embeds phrases ("Python programming", "deep learning", "natural language processing") as full sentences, so all six pairs produce scores. Expected ordering: `nlp` vs "natural language processing" and `tensorflow` vs "deep learning" high; `python` vs "Java" low; `python` vs "cooking" lowest.
- **GloVe fails loudly on phrases** — keyed-vector lookup of a multi-word string raises `KeyError`, which the `except` prints as `ERROR` lines; the same happens for casing or rare tokens outside the Twitter vocabulary. That OOV gap is the single strongest argument for context-aware models.

If `models` ends up empty, the cell prints setup guidance instead of crashing.

In [ ]:
if models:
    for name, model in models.items():
        print(f"\n=== {name} ===")
        for w1, w2, expected in test_pairs:
            try:
                if hasattr(model, 'encode'):
                    # SentenceTransformer
                    v1, v2 = model.encode([w1, w2])
                else:
                    # gensim keyed vectors
                    v1, v2 = model[w1], model[w2]
                sim = cosine_similarity([v1], [v2])[0][0]
                print(f"  '{w1:20s}' vs '{w2:20s}': {sim:.3f} (expected ~{expected})")
            except Exception as e:
                print(f"  '{w1:20s}' vs '{w2:20s}': ERROR — {e}")
else:
    print("No models available. This notebook works best after running notebooks 19, 21, and 22.")
    print("The benchmark methodology is defined — run those first, then return here.")

## Key Insight: Sentence Transformers > GloVe > Word2Vec for most resume tasks. But all have use cases.

**Benchmarking embeds the model choice in evidence — same pairs, same metric, every family.**

The measured story: sentence transformers score every pair (phrases, acronyms, casing included) while static GloVe vectors throw `KeyError` on anything outside their vocabulary — but GloVe stays useful when latency and memory matter and the vocabulary covers the domain. Ch. 24 changes the game again: zero-shot classification attacks a different problem (labeling text into categories) with NLI models, where no embedding-similarity benchmark applies at all.